In [ ]:

import sys, os
sys.path.append(os.pardir)
import numpy as np
from typing import Dict, List
import matplotlib.pyplot as plt

from libs.network import NeuralNet
from libs.trainer import Trainer
from libs.util import smooth_curve

# 设置字体为华文细黑
plt.rcParams['font.sans-serif'] = ['STHeiti']  # macOS 上的华文细黑
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

In [11]:
# 加载数据集
from dataset.mnist import load_mnist

(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, flatten=True, one_hot_label=True)

print(f"x_train: {x_train.shape}")
print(f"t_train: {t_train.shape}")
print(f"x_test: {x_test.shape}")
print(f"t_test: {t_test.shape}")

save_file: /Users/simeon/WorkSpace/Projects/Practice/DeepLearnStudy/example/../dataset/data/mnist.pkl
x_train: (60000, 784)
t_train: (60000, 10)
x_test: (10000, 784)
t_test: (10000, 10)


In [ ]:
# 分析 sigmoid 激活函数的问题
print("=== 分析 sigmoid 激活函数的问题 ===")

# 1. 测试 sigmoid 函数在大数值时的表现
import numpy as np
from libs.functions import sigmoid

# 测试不同范围的输入
test_inputs = [np.array([0.1, 0.5, 1.0, 5.0, 10.0, -10.0, -5.0])]
for inputs in test_inputs:
    outputs = sigmoid(inputs)
    print(f"输入: {inputs}")
    print(f"sigmoid输出: {outputs}")
    print(f"梯度 (1-sigmoid)*sigmoid: {(1.0 - outputs) * outputs}")
    print()

# 2. 检查权重初始化
print("\n=== 权重初始化检查 ===")
from libs.network import NeuralNet

# 创建小型网络进行测试
net_sigmoid = NeuralNet(
    input_size=784, 
    hidden_size_list=[100, 100], 
    output_size=10, 
    activation="sigmoid",
    weight_scale="xavier",
    verbose=False
)

net_relu = NeuralNet(
    input_size=784, 
    hidden_size_list=[100, 100], 
    output_size=10, 
    activation="relu",
    weight_scale="he",
    verbose=False
)

# 检查初始权重分布
print("Sigmoid网络权重统计:")
for key in ['W1', 'W2', 'W3']:
    w = net_sigmoid.params[key]
    print(f"{key}: mean={w.mean():.6f}, std={w.std():.6f}, min={w.min():.6f}, max={w.max():.6f}")

print("\nReLU网络权重统计:")
for key in ['W1', 'W2', 'W3']:
    w = net_relu.params[key]
    print(f"{key}: mean={w.mean():.6f}, std={w.std():.6f}, min={w.min():.6f}, max={w.max():.6f}")

=== 分析 sigmoid 激活函数的问题 ===
输入: [  0.1   0.5   1.    5.   10.  -10.   -5. ]
sigmoid输出: [5.24979187e-01 6.22459331e-01 7.31058579e-01 9.93307149e-01
 9.99954602e-01 4.53978687e-05 6.69285092e-03]
梯度 (1-sigmoid)*sigmoid: [2.49376040e-01 2.35003712e-01 1.96611933e-01 6.64805667e-03
 4.53958077e-05 4.53958077e-05 6.64805667e-03]


=== 权重初始化检查 ===
Sigmoid网络权重统计:
W1: mean=-0.000128, std=0.100290, min=-0.409855, max=0.425311
W2: mean=-0.000530, std=0.100968, min=-0.367390, max=0.386636
W3: mean=0.000206, std=0.095338, min=-0.315201, max=0.337895

ReLU网络权重统计:
W1: mean=-0.000411, std=0.141807, min=-0.628834, max=0.609631
W2: mean=0.000724, std=0.141791, min=-0.587460, max=0.486752
W3: mean=0.001308, std=0.137438, min=-0.448405, max=0.479530


In [13]:
# 3. 测试前向传播中的激活值
print("\n=== 前向传播中的激活值分析 ===")

# 取一小批数据进行测试
test_x = x_train[:10]  # 取前10个样本
test_t = t_train[:10]

print("测试 sigmoid 网络的前向传播:")
# 手动前向传播来观察每一层的输出
x = test_x
for i, layer_name in enumerate(net_sigmoid.layers.keys()):
    layer = net_sigmoid.layers[layer_name]
    x = layer.forward(x)
    if 'Sigmoid' in layer_name or 'Relu' in layer_name:
        print(f"{layer_name} 输出统计: mean={x.mean():.6f}, std={x.std():.6f}, min={x.min():.6f}, max={x.max():.6f}")
        print(f"  饱和神经元比例 (>0.99): {(x > 0.99).sum() / x.size * 100:.2f}%")
        print(f"  饱和神经元比例 (<0.01): {(x < 0.01).sum() / x.size * 100:.2f}%")
        
print("\n测试 ReLU 网络的前向传播:")
x = test_x
for i, layer_name in enumerate(net_relu.layers.keys()):
    layer = net_relu.layers[layer_name]
    x = layer.forward(x)
    if 'Sigmoid' in layer_name or 'Relu' in layer_name:
        print(f"{layer_name} 输出统计: mean={x.mean():.6f}, std={x.std():.6f}, min={x.min():.6f}, max={x.max():.6f}")
        print(f"  死亡神经元比例 (=0): {(x == 0).sum() / x.size * 100:.2f}%")

# 4. 测试初始损失和准确率
print("\n=== 初始性能对比 ===")
sigmoid_loss = net_sigmoid.loss(test_x, test_t)
sigmoid_acc = net_sigmoid.accuracy(test_x, test_t)
relu_loss = net_relu.loss(test_x, test_t)
relu_acc = net_relu.accuracy(test_x, test_t)

print(f"Sigmoid网络 - 初始损失: {sigmoid_loss:.4f}, 初始准确率: {sigmoid_acc:.4f}")
print(f"ReLU网络 - 初始损失: {relu_loss:.4f}, 初始准确率: {relu_acc:.4f}")

# 随机猜测的准确率应该是 1/10 = 0.1
print(f"随机猜测准确率: {1.0/10:.1f}")
print(f"Sigmoid网络是否接近随机猜测: {'是' if abs(sigmoid_acc - 0.1) < 0.02 else '否'}")


=== 前向传播中的激活值分析 ===
测试 sigmoid 网络的前向传播:
Sigmoid1 输出统计: mean=0.487460, std=0.188801, min=0.040988, max=0.951922
  饱和神经元比例 (>0.99): 0.00%
  饱和神经元比例 (<0.01): 0.00%
Sigmoid2 输出统计: mean=0.493805, std=0.126745, min=0.092150, max=0.840315
  饱和神经元比例 (>0.99): 0.00%
  饱和神经元比例 (<0.01): 0.00%

测试 ReLU 网络的前向传播:
Relu1 输出统计: mean=0.515605, std=0.745154, min=0.000000, max=4.244635
  死亡神经元比例 (=0): 50.30%
Relu2 输出统计: mean=0.537081, std=0.761566, min=0.000000, max=4.108185
  死亡神经元比例 (=0): 46.00%

=== 初始性能对比 ===
Sigmoid网络 - 初始损失: 23.6269, 初始准确率: 0.1000
ReLU网络 - 初始损失: 28.6801, 初始准确率: 0.1000
随机猜测准确率: 0.1
Sigmoid网络是否接近随机猜测: 是


In [14]:
# 5. 梯度分析
print("\n=== 梯度分析 ===")

# 计算梯度
sigmoid_grads = net_sigmoid.gradient(test_x, test_t)
relu_grads = net_relu.gradient(test_x, test_t)

print("Sigmoid网络梯度统计:")
for key in ['W1', 'W2', 'W3']:
    grad = sigmoid_grads[key]
    print(f"{key}: mean={grad.mean():.8f}, std={grad.std():.8f}, max_abs={np.abs(grad).max():.8f}")

print("\nReLU网络梯度统计:")
for key in ['W1', 'W2', 'W3']:
    grad = relu_grads[key]
    print(f"{key}: mean={grad.mean():.8f}, std={grad.std():.8f}, max_abs={np.abs(grad).max():.8f}")

# 检查梯度消失
print("\n梯度消失检查:")
for key in ['W1', 'W2', 'W3']:
    sigmoid_grad_norm = np.linalg.norm(sigmoid_grads[key])
    relu_grad_norm = np.linalg.norm(relu_grads[key])
    print(f"{key} 梯度范数 - Sigmoid: {sigmoid_grad_norm:.8f}, ReLU: {relu_grad_norm:.8f}")
    
    # 如果梯度范数过小，说明可能存在梯度消失
    if sigmoid_grad_norm < 1e-6:
        print(f"  警告: Sigmoid网络在{key}层可能存在梯度消失!")


=== 梯度分析 ===
Sigmoid网络梯度统计:
W1: mean=-0.00001597, std=0.00044253, max_abs=0.00416267
W2: mean=0.00025019, std=0.00339487, max_abs=0.01287006
W3: mean=0.00000000, std=0.04841676, max_abs=0.18574444

ReLU网络梯度统计:
W1: mean=0.00059504, std=0.01264879, max_abs=0.20084190
W2: mean=0.00297490, std=0.03654031, max_abs=0.39454221
W3: mean=0.00000000, std=0.12126760, max_abs=1.05471986

梯度消失检查:
W1 梯度范数 - Sigmoid: 0.12398802, ReLU: 3.54557849
W2 梯度范数 - Sigmoid: 0.34040797, ReLU: 3.66612107
W3 梯度范数 - Sigmoid: 1.53107250, ReLU: 3.83481807


## 问题分析和解决方案

基于上面的分析，sigmoid激活函数在深层网络中出现0.112左右准确率（接近随机猜测的0.1）的原因主要有：

### 1. 梯度消失问题
- **问题**: Sigmoid函数在输入值较大或较小时，梯度接近0，导致深层网络中的梯度消失
- **表现**: 从测试结果可以看到，sigmoid函数在±5以上时梯度几乎为0

### 2. 激活值饱和
- **问题**: Sigmoid输出范围是(0,1)，容易在深层网络中出现饱和现象
- **表现**: 大量神经元输出接近0或1，导致梯度消失

### 3. 权重初始化不当
- **问题**: 虽然使用了Xavier初始化，但对于深层sigmoid网络可能仍然不够
- **解决**: 可以尝试更小的初始化方差

### 4. 学习率问题
- **问题**: 对于sigmoid网络，可能需要更大的学习率来克服梯度消失
- **解决**: 尝试增加学习率或使用自适应学习率

### 建议的解决方案：
1. **使用更小的权重初始化**
2. **增加学习率**
3. **使用Batch Normalization**
4. **减少网络深度**
5. **使用残差连接（如果适用）**

In [15]:
# 解决方案测试
print("=== 解决方案测试 ===")

# 方案1: 使用更小的权重初始化
print("\n1. 测试更小的权重初始化")
net_sigmoid_small = NeuralNet(
    input_size=784, 
    hidden_size_list=[100, 100], 
    output_size=10, 
    activation="sigmoid",
    weight_scale=0.01,  # 使用更小的初始化
    verbose=False
)

# 测试初始性能
small_init_loss = net_sigmoid_small.loss(test_x, test_t)
small_init_acc = net_sigmoid_small.accuracy(test_x, test_t)
print(f"小权重初始化 - 初始损失: {small_init_loss:.4f}, 初始准确率: {small_init_acc:.4f}")

# 方案2: 使用Batch Normalization
print("\n2. 测试Batch Normalization")
net_sigmoid_bn = NeuralNet(
    input_size=784, 
    hidden_size_list=[100, 100], 
    output_size=10, 
    activation="sigmoid",
    weight_scale="xavier",
    use_batchnorm=True,  # 启用Batch Normalization
    verbose=False
)

bn_loss = net_sigmoid_bn.loss(test_x, test_t, train_flg=True)
bn_acc = net_sigmoid_bn.accuracy(test_x, test_t)
print(f"Batch Normalization - 初始损失: {bn_loss:.4f}, 初始准确率: {bn_acc:.4f}")

# 方案3: 减少网络深度
print("\n3. 测试浅层网络")
net_sigmoid_shallow = NeuralNet(
    input_size=784, 
    hidden_size_list=[100],  # 只有一个隐藏层
    output_size=10, 
    activation="sigmoid",
    weight_scale="xavier",
    verbose=False
)

shallow_loss = net_sigmoid_shallow.loss(test_x, test_t)
shallow_acc = net_sigmoid_shallow.accuracy(test_x, test_t)
print(f"浅层网络 - 初始损失: {shallow_loss:.4f}, 初始准确率: {shallow_acc:.4f}")

=== 解决方案测试 ===

1. 测试更小的权重初始化
小权重初始化 - 初始损失: 23.1232, 初始准确率: 0.1000

2. 测试Batch Normalization
Batch Normalization - 初始损失: 24.4750, 初始准确率: 0.1000

3. 测试浅层网络
浅层网络 - 初始损失: 23.2255, 初始准确率: 0.0000


In [16]:
# 简单训练测试
print("\n=== 简单训练测试 (5轮) ===")

# 使用较小的数据集进行快速测试
small_x_train = x_train[:1000]
small_t_train = t_train[:1000]
small_x_test = x_test[:200]
small_t_test = t_test[:200]

def quick_train_test(net, name, epochs=5):
    print(f"\n测试 {name}:")
    from libs.optimizer import SGD
    optimizer = SGD(lr=0.1)  # 使用较大的学习率
    
    for epoch in range(epochs):
        # 训练
        batch_size = 100
        for i in range(0, len(small_x_train), batch_size):
            batch_x = small_x_train[i:i+batch_size]
            batch_t = small_t_train[i:i+batch_size]
            
            grads = net.gradient(batch_x, batch_t)
            optimizer.update(net.params, grads)
        
        # 评估
        train_acc = net.accuracy(small_x_train, small_t_train)
        test_acc = net.accuracy(small_x_test, small_t_test)
        print(f"  Epoch {epoch+1}: 训练准确率={train_acc:.4f}, 测试准确率={test_acc:.4f}")

# 测试不同的解决方案
quick_train_test(net_sigmoid_bn, "Sigmoid + BatchNorm")
quick_train_test(net_sigmoid_shallow, "Sigmoid 浅层网络")

# 作为对比，测试ReLU
net_relu_small = NeuralNet(
    input_size=784, 
    hidden_size_list=[100], 
    output_size=10, 
    activation="relu",
    weight_scale="he",
    verbose=False
)
quick_train_test(net_relu_small, "ReLU 浅层网络")


=== 简单训练测试 (5轮) ===

测试 Sigmoid + BatchNorm:
  Epoch 1: 训练准确率=0.3530, 测试准确率=0.3000
  Epoch 2: 训练准确率=0.6500, 测试准确率=0.5250
  Epoch 3: 训练准确率=0.7060, 测试准确率=0.5700
  Epoch 4: 训练准确率=0.7410, 测试准确率=0.6200
  Epoch 5: 训练准确率=0.7700, 测试准确率=0.6700

测试 Sigmoid 浅层网络:
  Epoch 1: 训练准确率=0.2580, 测试准确率=0.2500
  Epoch 2: 训练准确率=0.3360, 测试准确率=0.3300
  Epoch 3: 训练准确率=0.4500, 测试准确率=0.4350
  Epoch 4: 训练准确率=0.5490, 测试准确率=0.4800
  Epoch 5: 训练准确率=0.6130, 测试准确率=0.5200

测试 ReLU 浅层网络:
  Epoch 1: 训练准确率=0.5700, 测试准确率=0.4450
  Epoch 2: 训练准确率=0.6980, 测试准确率=0.5800
  Epoch 3: 训练准确率=0.7700, 测试准确率=0.6550
  Epoch 4: 训练准确率=0.8020, 测试准确率=0.6950
  Epoch 5: 训练准确率=0.8310, 测试准确率=0.7250


## 总结

从上面的分析可以看出，sigmoid激活函数在深层网络中出现0.112左右准确率（接近随机猜测）的主要原因是：

### 核心问题
1. **梯度消失**: 从梯度分析可以看到，sigmoid网络的梯度范数明显小于ReLU网络
2. **权重初始化**: 即使使用Xavier初始化，对于深层sigmoid网络可能仍然不够
3. **激活函数饱和**: sigmoid函数在输入较大时梯度接近0

### 有效的解决方案
1. **Batch Normalization**: 这是最有效的解决方案，可以显著改善训练
2. **减少网络深度**: 浅层网络可以缓解梯度消失问题
3. **调整学习率**: 对于sigmoid网络可能需要更大的学习率
4. **权重初始化**: 使用更小的初始化值

### 建议
- 对于新项目，建议使用ReLU或其变种（LeakyReLU, ELU等）
- 如果必须使用sigmoid，建议配合Batch Normalization使用
- 考虑使用残差连接来缓解深层网络的训练问题

你的底层实现是正确的，问题出在sigmoid激活函数在深层网络中的固有特性上。